# Clean Consumption EIA Data

## Purpose
Data cleaning and transformation notebook that processes raw EIA (Energy Information Administration) residual fuel consumption data from the bronze layer and prepares it for analysis in the silver layer.

## Data Flow

**Data Source (Read):**
* `workspace.bronze.eia_residual_fuel_consumption_annual` - Raw EIA residual fuel consumption data

**Data Destination (Write):**
* `workspace.silver.eia_residual_fuel_consumption_annual` - Cleaned and standardized consumption data (overwrite mode)

## Workflow
1. **Extract** - Query bronze table for years 1980-2024
2. **Transform** - Select and rename columns:
   * `countryRegionName` → `Country`
   * `period` → `Year`
   * Keep: `productName`, `unitName`, `value`
3. **Load** - Write to silver schema with overwrite mode
4. **Verify** - Display sample of cleaned data

## Key Features
* **Time range filtering** - Focuses on 1980-2024 data for consistency
* **Column standardization** - Renames columns for downstream compatibility
* **Full refresh** - Overwrites silver table on each run to ensure data consistency
* **Validation** - Includes verification step to confirm successful load

In [0]:
# %pip install deltalake pandas pyarrow
from delta.tables import DeltaTable

# Load the Delta table metadata
pandas_df = spark.sql("""
    SELECT 
        countryRegionName as Country, 
        period as Year, 
        productName, 
        unitName,
        TRY_CAST(value AS FLOAT) as value
    FROM workspace.bronze.eia_residual_fuel_consumption_annual
    WHERE period BETWEEN 1980 AND 2024
""").toPandas()

#drop all rows with null values or zeros from the 'value' column
pandas_df = pandas_df.dropna(subset=['value'])
pandas_df = pandas_df[pandas_df['value'] != 0]

#convert pandas_df to spark_df
spark_df = spark.createDataFrame(pandas_df)

#write pandas_df to new delta table with schema 'silver'
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

spark_df.write.mode("overwrite").format("delta").saveAsTable(
    "silver.eia_residual_fuel_consumption_annual"
)

In [0]:
#verify the data
spark_df.show(10)

+--------------------+----+-----------------+--------------------+-----------+
|             Country|Year|      productName|            unitName|      value|
+--------------------+----+-----------------+--------------------+-----------+
|              Angola|2024|Residual fuel oil|thousand barrels ...|9.132891408|
|United Arab Emirates|2024|Residual fuel oil|thousand barrels ...|359.2068083|
|           Australia|2024|Residual fuel oil|thousand barrels ...|  15.480874|
|              Brazil|2024|Residual fuel oil|thousand barrels ...|86.61649315|
|              Canada|2024|Residual fuel oil|thousand barrels ...|   8.349727|
|               China|2024|Residual fuel oil|thousand barrels ...|  804.59728|
|             Germany|2024|Residual fuel oil|thousand barrels ...|  38.114754|
|             Algeria|2024|Residual fuel oil|thousand barrels ...|4.688240098|
|               Spain|2024|Residual fuel oil|thousand barrels ...| 151.308743|
|              France|2024|Residual fuel oil|thousan